# **Multi-agent System 튜토리얼**

**Multi-agent System** 이란 여러 AI 에이전트가 협력하여 복잡한 작업을 수행하는 시스템입니다.

```
사용자 요청
     │
     ▼
 [Supervisor / Orchestrator]
     │
     ├──▶ [Agent A: 리서처]  ──▶ 조사 결과
     ├──▶ [Agent B: 코더]    ──▶ 코드 생성
     └──▶ [Agent C: 작가]    ──▶ 문서 작성
                                     │
                                     ▼
                                최종 결과물
```

**주요 프레임워크:**
- **LangGraph**: 상태 기반 그래프 워크플로우, 복잡한 제어 흐름
- **AutoGen**: 에이전트 간 자율 대화, 팀 기반 협업


---
# **Part 1: LangGraph**

## **1. 실습환경 구성**

**1-1. 패키지 설치**

In [ ]:
!pip install langgraph langchain-openai langchain-core -q

**1-2. API 키 설정**

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "YOUR API KEY"


## **2. LangGraph 기초**

LangGraph는 **StateGraph** 를 사용해 노드(Node)와 엣지(Edge)로 워크플로우를 정의합니다.

```
[START] ──▶ [노드 A] ──▶ [노드 B] ──▶ [END]
```

핵심 개념:
- **State**: 노드 간 공유되는 상태 데이터
- **Node**: 상태를 입력받아 변환하는 함수
- **Edge**: 노드 간 연결 (일반 / 조건부)


**2-1. 기본 StateGraph**

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# State 정의: 노드 간 공유되는 데이터 구조
class SimpleState(TypedDict):
    messages: Annotated[list[str], operator.add]  # add = 리스트 누적
    step: int

def node_a(state: SimpleState) -> SimpleState:
    print(f'[노드 A] step: {state["step"]}')
    return {'messages': ['노드 A 처리 완료'], 'step': state['step'] + 1}

def node_b(state: SimpleState) -> SimpleState:
    print(f'[노드 B] step: {state["step"]}')
    return {'messages': ['노드 B 처리 완료'], 'step': state['step'] + 1}

def node_c(state: SimpleState) -> SimpleState:
    print(f'[노드 C] step: {state["step"]}')
    all_messages = '\n'.join(state['messages'])
    return {'messages': [f'최종 취합:\n{all_messages}'], 'step': state['step'] + 1}

# 그래프 구성
graph = StateGraph(SimpleState)
graph.add_node('A', node_a)
graph.add_node('B', node_b)
graph.add_node('C', node_c)
graph.set_entry_point('A')
graph.add_edge('A', 'B')
graph.add_edge('B', 'C')
graph.add_edge('C', END)

app = graph.compile()

result = app.invoke({'messages': [], 'step': 0})
print('\n=== 최종 상태 ===')
print(f'step: {result["step"]}')
for msg in result['messages']:
    print(f'  - {msg}')


**2-2. 조건부 엣지 (Conditional Edges)**

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, END

class ReviewState(TypedDict):
    content: str
    score: int
    feedback: str
    revision_count: int

def evaluate_node(state: ReviewState) -> ReviewState:
    content = state['content']
    score = min(100, len(content) * 2)
    feedback = '내용이 충분합니다.' if score >= 60 else '더 자세한 내용이 필요합니다.'
    print(f'[평가] 점수: {score}, 피드백: {feedback}')
    return {'score': score, 'feedback': feedback}

def improve_node(state: ReviewState) -> ReviewState:
    improved = state['content'] + ' (개선됨: 추가 내용이 보완되었습니다.)'
    print(f'[개선] 수정 {state["revision_count"]+1}회차 완료')
    return {'content': improved, 'revision_count': state['revision_count'] + 1}

def finalize_node(state: ReviewState) -> ReviewState:
    print(f'[완료] 최종 승인! 총 수정: {state["revision_count"]}회')
    return state

# 라우팅 함수 (조건부 엣지의 핵심)
def should_improve(state: ReviewState) -> Literal['improve', 'finalize']:
    if state['score'] < 60 and state['revision_count'] < 3:
        return 'improve'
    return 'finalize'

graph2 = StateGraph(ReviewState)
graph2.add_node('evaluate', evaluate_node)
graph2.add_node('improve', improve_node)
graph2.add_node('finalize', finalize_node)
graph2.set_entry_point('evaluate')
graph2.add_conditional_edges(
    'evaluate', should_improve,
    {'improve': 'improve', 'finalize': 'finalize'}
)
graph2.add_edge('improve', 'evaluate')  # 루프: 개선 후 재평가
graph2.add_edge('finalize', END)

app2 = graph2.compile()
result2 = app2.invoke({'content': 'AI 설명', 'score': 0, 'feedback': '', 'revision_count': 0})
print(f'\n최종 콘텐츠: {result2["content"][:80]}...')
print(f'최종 점수: {result2["score"]}')


## **3. 도구 사용 에이전트 (Tool-using Agent)**

LangGraph의 `create_react_agent`를 활용하여 **도구(Tool)를 사용하는 ReAct 에이전트**를 구축합니다.

**ReAct 패턴:**
```
Thought → Action (도구 호출) → Observation → Thought → ... → Final Answer
```


In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

@tool
def get_weather(city: str) -> str:
    'Returns current weather for a city.'
    weather_db = {
        '서울': '맑음, 기온 22°C, 습도 45%',
        '부산': '구름 많음, 기온 19°C, 습도 65%',
        '제주': '흐림, 기온 24°C, 습도 78%',
    }
    return weather_db.get(city, f'{city}의 날씨 정보가 없습니다.')

@tool
def calculate(expression: str) -> str:
    'Evaluates a math expression.'
    try:
        return f'{expression} = {eval(expression)}'
    except Exception as e:
        return f'계산 오류: {e}'

@tool
def search_workshop(query: str) -> str:
    'Searches Innocore workshop information.'
    info = {
        '일정': '2026년 6월 3-5일, 이노코어 본사 대회의실',
        '강사': '이준형 (LLM 특강)',
        '내용': 'LangChain, RAG, Multi-agent 시스템 실습',
    }
    for key, value in info.items():
        if key in query:
            return f'[{key}] {value}'
    return str(list(info.values()))

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
agent = create_react_agent(llm, tools=[get_weather, calculate, search_workshop])

queries = [
    '서울과 제주의 날씨를 비교해주세요.',
    '워크숍 일정이 어떻게 되나요?',
    '서울 기온(22)과 제주 기온(24)의 평균을 계산해주세요.',
]

for query in queries:
    print(f'Q: {query}')
    result = agent.invoke({'messages': [HumanMessage(content=query)]})
    print(f'A: {result["messages"][-1].content}')
    print('-' * 60)


## **4. Multi-agent: 리서처 + 작가 패턴**

두 개의 특화된 에이전트가 협력하는 패턴을 구현합니다.

```
사용자 요청
     │
     ▼
[Researcher] ── 조사결과 ──▶ [Writer] ── 최종 글 ──▶ END
```


In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import operator

class MultiAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    topic: str
    research_result: str
    final_article: str

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)

def researcher_node(state: MultiAgentState) -> MultiAgentState:
    print('\n[리서처] 조사 중...')
    prompt = (
        f'당신은 전문 리서처입니다.\n'
        f'주제: {state["topic"]}\n\n'
        '핵심 개념, 특징, 활용 사례를 300자 이내로 구조화해서 정리하세요.'
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f'[리서처] 완료 ({len(response.content)}자)')
    return {
        'messages': [AIMessage(content=f'[리서처 조사결과]\n{response.content}')],
        'research_result': response.content
    }

def writer_node(state: MultiAgentState) -> MultiAgentState:
    print('\n[작가] 글 작성 중...')
    prompt = (
        f'당신은 전문 테크 라이터입니다.\n'
        f'주제: {state["topic"]}\n\n'
        f'리서치 결과:\n{state["research_result"]}\n\n'
        '위 정보를 바탕으로 읽기 쉬운 글을 400자 이내로 작성하세요. 마지막에 한 줄 요약 포함.'
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f'[작가] 완료 ({len(response.content)}자)')
    return {
        'messages': [AIMessage(content=f'[최종 글]\n{response.content}')],
        'final_article': response.content
    }

graph3 = StateGraph(MultiAgentState)
graph3.add_node('researcher', researcher_node)
graph3.add_node('writer', writer_node)
graph3.set_entry_point('researcher')
graph3.add_edge('researcher', 'writer')
graph3.add_edge('writer', END)
multi_agent_app = graph3.compile()


In [ ]:
topic = 'LangGraph를 활용한 멀티에이전트 시스템 구축'
print(f'주제: {topic}\n' + '='*60)

result = multi_agent_app.invoke({
    'messages': [HumanMessage(content=topic)],
    'topic': topic,
    'research_result': '',
    'final_article': '',
})

print('\n' + '='*60)
print('=== 최종 결과 ===')
print(result['final_article'])


## **5. Supervisor 패턴 (고급)**

Supervisor 에이전트가 상황을 판단하고 적절한 워커에게 라우팅합니다.

```
사용자 요청 ──▶ [Supervisor]
                    │
         ┌──────────┼──────────┐
         ▼          ▼          ▼
   [Researcher] [Coder]  [Analyst]
         └──────────┴──────────┘
                    │ 결과 취합
                    ▼
             [Supervisor] → FINISH
```


In [ ]:
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import operator

WORKERS = ['researcher', 'coder', 'analyst']

class SupervisorState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    next_worker: str

llm_sup = ChatOpenAI(model='gpt-4o-mini', temperature=0)

def supervisor_node(state: SupervisorState) -> SupervisorState:
    system = (
        'You are a team supervisor. Reply with ONLY one word:\n'
        '- researcher: when information research is needed\n'
        '- coder: when code writing/analysis is needed\n'
        '- analyst: when data analysis/summary is needed\n'
        '- FINISH: when the task is complete\n'
        'Look at the conversation and decide the next worker.'
    )
    messages = [SystemMessage(content=system)] + state['messages']
    response = llm_sup.invoke(messages)
    decision = response.content.strip()
    if decision not in WORKERS and decision != 'FINISH':
        decision = 'FINISH'
    print(f'\n[Supervisor] 결정: {decision}')
    return {
        'messages': [AIMessage(content=f'[Supervisor → {decision}]')],
        'next_worker': decision
    }

def make_worker(name, role_kr):
    def worker_node(state: SupervisorState) -> SupervisorState:
        print(f'\n[{name}] 작업 중...')
        last_human = next(
            (m.content for m in reversed(state['messages']) if isinstance(m, HumanMessage)),
            '작업 수행'
        )
        prompt = f'당신은 {role_kr}입니다. 다음 요청을 처리하세요: {last_human}'
        response = llm_sup.invoke([HumanMessage(content=prompt)])
        print(f'[{name}] 완료')
        return {'messages': [AIMessage(content=f'[{name.upper()}]\n{response.content}')]}
    worker_node.__name__ = f'{name}_node'
    return worker_node

researcher_node2 = make_worker('researcher', '전문 리서처, 핵심 정보를 조사하고 정리')
coder_node2 = make_worker('coder', '시니어 개발자, 파이썬 코드를 작성하고 설명')
analyst_node2 = make_worker('analyst', '데이터 분석가, 정보를 분석하고 인사이트 제공')

def route_to_worker(state: SupervisorState) -> str:
    worker = state.get('next_worker', 'FINISH')
    return '__end__' if worker == 'FINISH' else worker

graph4 = StateGraph(SupervisorState)
graph4.add_node('supervisor', supervisor_node)
graph4.add_node('researcher', researcher_node2)
graph4.add_node('coder', coder_node2)
graph4.add_node('analyst', analyst_node2)
graph4.set_entry_point('supervisor')
graph4.add_conditional_edges(
    'supervisor', route_to_worker,
    {'researcher': 'researcher', 'coder': 'coder', 'analyst': 'analyst', '__end__': END}
)
for w in WORKERS:
    graph4.add_edge(w, 'supervisor')  # 워커 완료 후 supervisor로 복귀

supervisor_app = graph4.compile()


In [ ]:
task = 'Python으로 간단한 RAG 시스템을 구현하는 방법을 알려주세요.'
print(f'작업: {task}\n' + '='*60)

result = supervisor_app.invoke(
    {'messages': [HumanMessage(content=task)], 'next_worker': ''},
    {'recursion_limit': 8}
)

print('\n' + '='*60)
print('=== 전체 대화 로그 ===')
for msg in result['messages']:
    if isinstance(msg, AIMessage):
        print(f'\n{msg.content[:250]}...')


---
# **Part 2: AutoGen**

**AutoGen** 은 Microsoft에서 개발한 멀티에이전트 프레임워크입니다.
에이전트 간 **자율적인 대화**를 통해 복잡한 작업을 수행합니다.

**LangGraph vs AutoGen 비교:**

| 특징 | LangGraph | AutoGen |
|------|-----------|--------|
| 방식 | 그래프 기반 워크플로우 | 에이전트 간 대화 |
| 제어 | 명시적 상태 관리 | 자율적 협상 |
| 적합 | 예측 가능한 파이프라인 | 탐색적 문제 해결 |
| 학습 곡선 | 중간 | 낮음 |


## **6. 실습환경 구성 (AutoGen)**

**6-1. 패키지 설치**

In [ ]:
!pip install autogen-agentchat autogen-ext[openai] nest_asyncio -q

**6-2. 초기 설정**

In [ ]:
import os
import asyncio
import nest_asyncio
nest_asyncio.apply()  # Colab 비동기 환경 설정

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    api_key=os.environ.get('OPENAI_API_KEY')
)
print('AutoGen 설정 완료!')


## **7. 기본 에이전트 대화**

**7-1. 두 에이전트 대화 (RoundRobinGroupChat)**

In [ ]:
researcher_ag = AssistantAgent(
    name='Researcher',
    model_client=model_client,
    system_message=(
        '당신은 전문 리서처입니다.\n'
        '주어진 주제를 조사하고 핵심 정보를 3-4가지로 정리하세요.\n'
        '조사가 완료되면 Writer에게 전달하세요.'
    ),
)

writer_ag = AssistantAgent(
    name='Writer',
    model_client=model_client,
    system_message=(
        '당신은 전문 작가입니다.\n'
        'Researcher가 제공한 정보를 바탕으로 읽기 쉬운 글을 작성하세요.\n'
        '글이 완성되면 반드시 TERMINATE 라고 마무리하세요.'
    ),
)

termination = TextMentionTermination('TERMINATE') | MaxMessageTermination(6)
team = RoundRobinGroupChat([researcher_ag, writer_ag], termination_condition=termination)

async def run_team():
    task = '멀티에이전트 AI 시스템의 현재 동향과 미래 전망에 대해 알려주세요.'
    print(f'작업: {task}\n' + '='*60)
    await Console(team.run_stream(task=task))

asyncio.run(run_team())


**7-2. 팀 리셋 후 재사용**

In [ ]:
async def run_again():
    await team.reset()
    task = 'Python과 JavaScript 중 AI 개발에 더 적합한 언어는 무엇인가요?'
    print(f'작업: {task}\n' + '='*60)
    await Console(team.run_stream(task=task))

asyncio.run(run_again())


## **8. 그룹챗 (SelectorGroupChat)**

**SelectorGroupChat**: LLM이 다음 발화자를 동적으로 선택합니다.

```
사용자 요청
     │
     ▼
[SelectorGroupChat]
  LLM이 판단: 누가 답해야 하나?
  ↓           ↓           ↓
[PM]      [Developer]  [Designer]
```


In [ ]:
pm_agent = AssistantAgent(
    name='ProjectManager',
    model_client=model_client,
    system_message='당신은 프로젝트 매니저입니다. 프로젝트 계획, 일정, 리소스 관리에 전문입니다.',
)

dev_agent = AssistantAgent(
    name='Developer',
    model_client=model_client,
    system_message='당신은 시니어 개발자입니다. 기술적인 구현, 코드, 아키텍처에 전문입니다.',
)

qa_agent = AssistantAgent(
    name='QAEngineer',
    model_client=model_client,
    system_message='당신은 QA 엔지니어입니다. 테스트 전략, 품질 보증, 버그 예방에 전문입니다.',
)

selector_team = SelectorGroupChat(
    [pm_agent, dev_agent, qa_agent],
    model_client=model_client,
    termination_condition=MaxMessageTermination(8),
)

async def run_selector():
    task = '회사 내부 문서를 검색하는 RAG 챗봇을 2주 안에 개발해야 합니다. 어떻게 접근하면 좋을까요?'
    print(f'작업: {task}\n' + '='*60)
    await Console(selector_team.run_stream(task=task))

asyncio.run(run_selector())


## **9. 도구(Tool) 사용 AutoGen 에이전트**

In [ ]:
import datetime

def get_current_date() -> str:
    'Returns current date and time.'
    return datetime.datetime.now().strftime('%Y년 %m월 %d일 %H:%M')

def calculate_days_until(target_date: str) -> str:
    'Calculates days until a target date (YYYY-MM-DD format).'
    try:
        target = datetime.datetime.strptime(target_date, '%Y-%m-%d')
        delta = (target - datetime.datetime.now()).days
        if delta > 0:
            return f'{target_date}까지 {delta}일 남았습니다.'
        elif delta == 0:
            return '오늘입니다!'
        else:
            return f'{abs(delta)}일 전에 지났습니다.'
    except ValueError:
        return '날짜 형식 오류. YYYY-MM-DD 형식으로 입력하세요.'

def get_workshop_schedule() -> str:
    'Returns Innocore workshop schedule.'
    return (
        '이노코어 워크숍 2026 일정:\n'
        '- Day 1 (2026-06-03): LangChain 기초\n'
        '- Day 2 (2026-06-04): RAG 실습\n'
        '- Day 3 (2026-06-05): Multi-agent 시스템\n'
        '장소: 이노코어 본사 대회의실 (서울시 강남구)'
    )

schedule_agent = AssistantAgent(
    name='ScheduleAssistant',
    model_client=model_client,
    tools=[get_current_date, calculate_days_until, get_workshop_schedule],
    system_message=(
        '당신은 일정 관리 어시스턴트입니다.\n'
        '제공된 도구를 활용하여 일정 관련 질문에 답하세요.\n'
        '모든 정보를 확인한 후 TERMINATE 로 종료하세요.'
    ),
)

tool_team = RoundRobinGroupChat(
    [schedule_agent],
    termination_condition=TextMentionTermination('TERMINATE') | MaxMessageTermination(4),
)

async def run_tool_team():
    task = '오늘 날짜를 확인하고, 워크숍 일정을 알려주세요. 각 날짜까지 며칠 남았는지도 계산해주세요.'
    print(f'작업: {task}\n' + '='*60)
    await Console(tool_team.run_stream(task=task))

asyncio.run(run_tool_team())


## **정리**

### LangGraph vs AutoGen 선택 가이드

| 상황 | 추천 프레임워크 |
|------|----------------|
| 정해진 워크플로우 실행 | LangGraph |
| 조건부 분기 / 루프 처리 | LangGraph |
| 자유로운 에이전트 대화 | AutoGen |
| 동적 역할 배분 | AutoGen |
| 빠른 프로토타이핑 | AutoGen |

### 실무 활용 패턴 (조합)

```
[사용자 입력]
    │
    ▼
[LangGraph: 전처리 & 라우팅]
    │
    ├──▶ [AutoGen 팀: 복잡한 추론 필요 시]
    │         ├── Researcher
    │         └── Writer
    ├──▶ [RAG 파이프라인: 지식 검색 필요 시]
    └──▶ [단순 LLM 호출: 빠른 응답 필요 시]
              │
              ▼
        [최종 응답 생성]
```
